In [47]:
import json
import numpy as np
import os
import os.path as osp
import pandas as pd
import torch
import ast
import gzip

from collections import defaultdict
from torch_geometric.data import extract_zip
from torch_geometric.data import HeteroData
from sentence_transformers import SentenceTransformer
from torch_geometric.io import fs


In [3]:
def train_test_split(path,seq_len=15):
    users_id = []
    df_types = ['train', 'valid', 'test']
    seqs = {type: defaultdict(list) for type in df_types}
    with open(os.path.join(path, 'sequential_data.txt'), 'r') as file:
        for row in file:
            row_lst = list(map(int, row.strip().split()))
            users_id.append(row_lst[0])
            items = [i-1 for i in row_lst[1:]]
            items_types = {'train':items[:-2], 'valid': items[-(seq_len + 2):-2], 'test': items[-(seq_len+1):-1]}
            for tp in df_types:
                cur_items = items_types[tp]
                if tp != 'train':
                    gaps = [-1] * (seq_len - len(cur_items))
                    seqs[tp]['item_ID'].append(cur_items + gaps)
                else:
                    seqs[tp]['item_ID'].append(cur_items)
                seqs[tp]['item_ID_next'].append(items[-1 if tp =='test' else -2])
        for tp in df_types:
            seqs[tp]['user_ID'] = users_id
            seqs[tp] = pd.DataFrame(seqs[tp])
        return seqs



In [4]:
path = r'C:\datasets\beauty'
dfs = train_test_split(path)

In [5]:
train, val, test = dfs['train'], dfs['valid'], dfs['test']

In [6]:
train.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2]",3,1
1,"[5, 6, 7, 8, 9]",3,2
2,"[3, 11, 12, 13, 14, 15, 16]",17,3
3,"[19, 20, 21, 22]",3,4
4,"[3, 24, 25, 26, 27, 28, 29]",30,5
5,"[32, 33, 34, 3, 35, 36, 37, 38, 39, 40, 41, 42...",47,6
6,"[49, 50, 51, 52, 53]",54,7
7,"[55, 56, 57]",3,8
8,"[59, 60, 61, 21, 62, 63, 64, 65, 66, 67, 68, 6...",81,9
9,"[83, 84, 85, 82, 86, 58, 87, 88]",89,10


In [7]:
val.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2, -1, -1, -1, -1, -1, -1, -1, -1, -1, ...",3,1
1,"[5, 6, 7, 8, 9, -1, -1, -1, -1, -1, -1, -1, -1...",3,2
2,"[3, 11, 12, 13, 14, 15, 16, -1, -1, -1, -1, -1...",17,3
3,"[19, 20, 21, 22, -1, -1, -1, -1, -1, -1, -1, -...",3,4
4,"[3, 24, 25, 26, 27, 28, 29, -1, -1, -1, -1, -1...",30,5
5,"[33, 34, 3, 35, 36, 37, 38, 39, 40, 41, 42, 43...",47,6
6,"[49, 50, 51, 52, 53, -1, -1, -1, -1, -1, -1, -...",54,7
7,"[55, 56, 57, -1, -1, -1, -1, -1, -1, -1, -1, -...",3,8
8,"[66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 7...",81,9
9,"[83, 84, 85, 82, 86, 58, 87, 88, -1, -1, -1, -...",89,10


In [8]:
test.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2, 3, -1, -1, -1, -1, -1, -1, -1, -1, -...",4,1
1,"[5, 6, 7, 8, 9, 3, -1, -1, -1, -1, -1, -1, -1,...",10,2
2,"[3, 11, 12, 13, 14, 15, 16, 17, -1, -1, -1, -1...",18,3
3,"[19, 20, 21, 22, 3, -1, -1, -1, -1, -1, -1, -1...",23,4
4,"[3, 24, 25, 26, 27, 28, 29, 30, -1, -1, -1, -1...",31,5
5,"[34, 3, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44...",48,6
6,"[49, 50, 51, 52, 53, 54, -1, -1, -1, -1, -1, -...",3,7
7,"[55, 56, 57, 3, -1, -1, -1, -1, -1, -1, -1, -1...",58,8
8,"[67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 7...",82,9
9,"[83, 84, 85, 82, 86, 58, 87, 88, 89, -1, -1, -...",90,10


In [9]:
train.columns

Index(['item_ID', 'item_ID_next', 'user_ID'], dtype='object')

In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22363 entries, 0 to 22362
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   item_ID       22363 non-null  object
 1   item_ID_next  22363 non-null  int64 
 2   user_ID       22363 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 524.3+ KB


In [11]:
train = train.iloc[:,[2,1,0]]
val = val.iloc[:, [2,1,0]]
test = test.iloc[:, [2,1,0]]
train

,user_ID,item_ID_next,item_ID
0,1,3,"[0, 1, 2]"
1,2,3,"[5, 6, 7, 8, 9]"
2,3,17,"[3, 11, 12, 13, 14, 15, 16]"
3,4,3,"[19, 20, 21, 22]"
4,5,30,"[3, 24, 25, 26, 27, 28, 29]"
...,...,...,...
22358,22359,11811,"[11793, 11810, 12041]"
22359,22360,3024,"[10746, 3022, 5594, 6465, 9743]"
22360,22361,11795,"[12054, 11802, 9267]"
22361,22362,3033,"[3022, 9743, 10606]"


In [12]:
train.isnull().sum()

user_ID         0
item_ID_next    0
item_ID         0
dtype: int64

In [13]:
val.isnull().sum()

user_ID         0
item_ID_next    0
item_ID         0
dtype: int64

In [14]:
def df_to_dict_tensor(df, cols):
        res = {}
        for col in cols:
            if isinstance(df[col].iloc[0], list):
                if df[col].apply(len).nunique() == 1:
                    res[col] = torch.tensor(df[col].to_list(), dtype = torch.int64)
                else:
                    res[col] = df['item_ID'].to_list()
            else:
                res[col] = torch.tensor(df[col].to_numpy())
        for col in cols:
            next_col = col + '_next'
            if next_col in df.columns:
                res[next_col] = torch.tensor(df[next_col].to_numpy())
        res['user_ID'] = torch.from_numpy(df['user_ID'].to_numpy())
        return res



In [15]:
model = SentenceTransformer('sentence-transformers/sentence-t5-xl')

2_Dense/model.safetensors:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

C:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--sentence-t5-xl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [30]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [32]:
def encode(text, model = model, device = device):
    model = model.to(device)
    embeddings = model.encode(batch_size=2, sentences=text, show_progress_bar=True, convert_to_tensor=True, normalize_embeddings=False)
    return embeddings

In [34]:
def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield eval(l)

In [ ]:
df_graph = HeteroData()
with open(os.path.join(path,'datamaps.json'), 'r') as file:
    maps = json.load(file)
hist = {df_type: df_to_dict_tensor(ds, ['item_ID'])
        for df_type, ds in dfs.items()}
df_graph['user', 'rated', 'item'].history = hist
asin2id = pd.DataFrame([{"asin": key, "id": int(val) - 1} for key, val in maps["item2id"].items()])
item_data = pd.DataFrame([json.dumps(meta) for meta in parse(path=os.path.join(path, "meta.json.gz"))]).merge(asin2id, on="asin").sort_values(by="id").fillna({"brand": "Unknown"})
sentences = item_data.apply(
        lambda row:
            "Title: " +
            str(row["title"]) + "; " +
            "Brand: " +
            str(row["brand"]) + "; " +
            "Categories: " +
             str(row["categories"][0]) + "; " +
             "Price: " +
             str(row["price"]) + "; ",
             axis=1)
item_emb = encode(sentences)
df_graph['item'].x = item_emb
df_graph['item'].text = np.array(sentences)
gen = torch.Generator()
gen.manual_seed(42)
df_graph['item'].is_train = torch.rand(item_emb.shape[0], generator=gen) > 0.05

In [ ]:
maps.keys()

In [ ]:
list(maps['user2id'].items())[:20]

In [ ]:
len(list(maps['user2id'].items()))

In [ ]:
item_data

In [ ]:
id_s = item_data.id
id_s

In [ ]:
np.dstack((id_s, np.array(sentences)))

In [ ]:
df_graph['item'].x = item_emb
df_graph['item'].text = np.dstack((id_s, np.array(sentences)))
gen = torch.Generator()
gen.manual_seed(42)
df_graph['item'].is_train = torch.rand(item_emb.shape[0], generator=gen) > 0.05

In [ ]:
item_emb

In [ ]:
df_graph

In [ ]:
torch.save(df_graph, 'heterodata_object.pt')

In [ ]:
loaded_data = torch.load('heterodata_object.pt', weights_only=False)
loaded_data